### 1. Install Dependencies
We need `f5-tts`, `pypdf` for text extraction, and `pydub` for audio manipulation.

In [7]:
!pip install -U f5-tts pypdf pydub

### 2. File Upload and Preprocessing
Run this cell to upload your PDF and the reference voice audio (mp3/wav).

In [8]:
import os
import sys
# Force refresh of the site-packages if needed
import pkg_resources
import re
from google.colab import files

# Standard imports
from pypdf import PdfReader
from pydub import AudioSegment

# Create directory for chunks
os.makedirs('chunks', exist_ok=True)

print("Upload your PDF file:")
uploaded_pdf = files.upload()
pdf_path = list(uploaded_pdf.keys())[0]

print("\nUpload your Reference Audio file:")
uploaded_audio = files.upload()
ref_audio_path = list(uploaded_audio.keys())[0]

# 1. Process PDF: Extract text and split by full stops
reader = PdfReader(pdf_path)
full_text = ""
for page in reader.pages:
    full_text += page.extract_text() + " "

# Split text by full stops, exclamation marks, or question marks
sentences = [s.strip() for s in re.split(r'[.!?]', full_text) if s.strip()]

# 2. Process Audio: Cut to first 5 seconds to avoid bugs
audio = AudioSegment.from_file(ref_audio_path)
trimmed_audio = audio[:5000] # 5 seconds
trimmed_ref_path = "trimmed_ref.wav"
trimmed_audio.export(trimmed_ref_path, format="wav")

print(f"\nPrepared {len(sentences)} sentences for processing.")
print(f"Reference audio trimmed to 5 seconds.")

Upload your PDF file:


/tmp/ipykernel_2323/4130513768.py:4: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


Saving Jake_Sully.pdf to Jake_Sully (1).pdf

Upload your Reference Audio file:


Saving my voice 5 second.mp3 to my voice 5 second (1).mp3

Prepared 738 sentences for processing.
Reference audio trimmed to 5 seconds.


### 3. Generate Audio using F5-TTS
We will process each sentence individually and save them in the `chunks` folder indexed correctly.

In [11]:
from f5_tts.api import F5TTS
import torch
import soundfile as sf

# Initialize model
device = "cuda" if torch.cuda.is_available() else "cpu"
f5tts = F5TTS(device=device)

print(f"Using device: {device}")
print("Starting generation...")

valid_sentences = [s for s in sentences if len(s.strip()) > 2]
chunk_paths = []

for i, text in enumerate(valid_sentences):
    output_path = f"chunks/chunk_{i:04d}.wav"

    if i % 5 == 0 or i == len(valid_sentences)-1:
        print(f"[{i+1}/{len(valid_sentences)}] Generating: {text[:60]}...")

    try:
        # The infer method returns (wav, sr, spect)
        wav, sr, _ = f5tts.infer(
            gen_text=text,
            ref_file=trimmed_ref_path,
            ref_text=""
        )

        # Save the returned audio data to a file
        sf.write(output_path, wav, sr)
        chunk_paths.append(output_path)

    except Exception as e:
        print(f"Error on sentence {i}: {e}")
        continue

print(f"\nSuccess! {len(chunk_paths)} chunks generated.")

Download Vocos from huggingface charactr/vocos-mel-24khz

vocab :  /usr/local/lib/python3.12/dist-packages/f5_tts/infer/examples/vocab.txt
token :  custom
model :  /root/.cache/huggingface/hub/models--SWivid--F5-TTS/snapshots/84e5a410d9cead4de2f847e7c9369a6440bdfaca/F5TTS_v1_Base/model_1250000.safetensors 

Using device: cuda
Starting generation...
[1/737] Generating: Jake Sully wakes up in icy darkness inside a metal coffin...
Converting audio...
No reference text provided, transcribing reference audio...


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

...

[736/737] Generating: Jake's eyes open...
Converting audio...
Using cached preprocessed reference audio...
Using cached reference text...

ref_text   Why are homeowners replacing their air conditioners?. 
gen_text 0 Jake's eyes open


Generating audio in 1 batches...


100%|██████████| 1/1 [00:01<00:00,  1.65s/it]


[737/737] Generating: He is finally home...
Converting audio...
Using cached preprocessed reference audio...
Using cached reference text...

ref_text   Why are homeowners replacing their air conditioners?. 
gen_text 0 He is finally home


Generating audio in 1 batches...


100%|██████████| 1/1 [00:01<00:00,  1.70s/it]


Success! 737 chunks generated.


### 4. Merge Chunks into Final Audio
Finally, we combine all indexed chunks into one single file.

In [12]:
combined = AudioSegment.empty()

for path in sorted(chunk_paths):
    segment = AudioSegment.from_wav(path)
    combined += segment + AudioSegment.silent(duration=300) # Small pause between sentences

final_output = "final_voice_cloned_audio.wav"
combined.export(final_output, format="wav")

print(f"Final audio generated: {final_output}")
files.download(final_output)

Final audio generated: final_voice_cloned_audio.wav


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 5. Export System to GitHub
Run this cell to package the code into a format ready for GitHub.

In [13]:
import os

# 1. Create the application script
app_code = """
import os
import re
from pypdf import PdfReader
from pydub import AudioSegment
from f5_tts.api import F5TTS
import torch
import soundfile as sf

def run_tts_system(pdf_path, ref_audio_path):
    os.makedirs('chunks', exist_ok=True)

    # Process PDF
    reader = PdfReader(pdf_path)
    sentences = [s.strip() for s in re.split(r'[.!?]', " ".join([p.extract_text() for p in reader.pages])) if s.strip()]

    # Trim Audio (5s)
    audio = AudioSegment.from_file(ref_audio_path)[:5000]
    audio.export("trimmed_ref.wav", format="wav")

    # Initialize AI
    f5tts = F5TTS(device="cuda" if torch.cuda.is_available() else "cpu")

    chunk_paths = []
    for i, text in enumerate(sentences):
        out = f"chunks/chunk_{i:04d}.wav"
        wav, sr, _ = f5tts.infer(gen_text=text, ref_file="trimmed_ref.wav", ref_text="")
        sf.write(out, wav, sr)
        chunk_paths.append(out)

    # Merge
    combined = AudioSegment.empty()
    for p in sorted(chunk_paths):
        combined += AudioSegment.from_wav(p) + AudioSegment.silent(duration=300)
    combined.export("output_full.wav", format="wav")
    print("Done!")

if __name__ == '__main__':
    # You can customize these paths
    run_tts_system('input.pdf', 'reference.mp3')
"""

with open('app.py', 'w') as f: f.write(app_code)

# 2. Create requirements.txt
with open('requirements.txt', 'w') as f:
    f.write("f5-tts\npypdf\npydub\ntorch\nsoundfile")

print("System files 'app.py' and 'requirements.txt' are ready.")

System files 'app.py' and 'requirements.txt' are ready.


To push to GitHub, run the following (Replace placeholders with your info):

In [17]:
# @title GitHub Push Config (Forcing Upload)
GITHUB_USER = "kavimarkstar"
GITHUB_REPO = "F5-TTS-English"
GITHUB_TOKEN = "REMOVED_FOR_SECURITY"

# Configure Git
!git config --global user.email "you@example.com"
!git config --global user.name "{GITHUB_USER}"

# Set up the remote
!git remote remove origin 2>/dev/null
!git remote add origin https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git

# Force push to overwrite the remote with our local files
!git branch -M main
!git push -u origin main --force

Branch 'main' set up to track remote branch 'main' from 'origin'.
Everything up-to-date


### 6. Push the Notebook itself to GitHub

In [ ]:
import json
import os
from google.colab import _message

# 1. Get current notebook content
notebook_json = _message.blocking_request('get_ipynb', request='', timeout_sec=30)
notebook_data = notebook_json['ipynb']

# 2. Scrub the token string completely from the JSON structure
TOKEN_TO_REMOVE = "REMOVED_FOR_SECURITY"
for cell in notebook_data.get('cells', []):
    if cell.get('cell_type') == 'code':
        cell['source'] = [line.replace(TOKEN_TO_REMOVE, 'REMOVED_FOR_SECURITY') for line in cell['source']]

# 3. Save the clean file
notebook_filename = "F5_TTS_Voice_Cloning.ipynb"
with open(notebook_filename, 'w', encoding='utf-8') as f:
    json.dump(notebook_data, f, indent=1, ensure_ascii=False)

# 4. HARD RESET: Delete git history and push fresh to bypass Push Protection
!rm -rf .git
!git init
!git config --global user.email "you@example.com"
!git config --global user.name "kavimarkstar"
!git remote add origin https://{TOKEN_TO_REMOVE}@github.com/kavimarkstar/F5-TTS-English.git
!git checkout -b main
!git add "{notebook_filename}" "app.py" "requirements.txt"
!git commit -m "Initial clean upload"
!git push -u origin main --force

print(f"\nSuccess! Your clean notebook '{notebook_filename}' is now on GitHub.")